# Kaggle ResNet18 SimCLR: resnet18_imagenet_covidqu_syn

This notebook runs only `resnet18_imagenet_covidqu_syn` (ImageNet initialized SimCLR on Stage 1 DCGAN synthetic COVID-QU-Syn). Pretraining and fine-tuning are separate so multiple Kaggle sessions can run/resume independently.


## 1. Enable GPU

In Kaggle, open notebook settings and set Accelerator to GPU. Do not use TPU for this pipeline.

In [19]:
from pathlib import Path
import os
import shutil
import pandas as pd

print('Kaggle input exists:', Path('/kaggle/input').exists())
print('Kaggle working exists:', Path('/kaggle/working').exists())
!nvidia-smi

Kaggle input exists: True
Kaggle working exists: True
Mon Jun  1 17:42:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  

## 2. Clone or Pull Repository

In [20]:
REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('REPO_ROOT:', REPO_ROOT)
!git rev-parse --short HEAD

/kaggle/working/contrastive-synthesis-medcls_CVProject
Already up to date.
REPO_ROOT: /kaggle/working/contrastive-synthesis-medcls_CVProject
e174c19d


## 3. Install Minimal Dependencies

Kaggle already includes PyTorch. Install only lightweight packages used by the scripts.

In [21]:
!pip install -q timm scikit-learn matplotlib pandas Pillow

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA available: True


## 4. Edit Kaggle Dataset Paths

After adding your Kaggle Dataset to this notebook, edit these paths to match the folder names under `/kaggle/input`. The helper cell below links them into the repository as `data/processed/...`, so the existing scripts do not need path changes.

In [22]:
from pathlib import Path
import os

DATA_ROOT = Path("/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data")

LABELLED_SOURCE = DATA_ROOT / "processed/labelled_4232"
UNLABELLED_SOURCE = DATA_ROOT / "processed/unlabelled_16934"
SYNTHETIC_SOURCE = DATA_ROOT / "processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan"
MANIFEST_SOURCE = DATA_ROOT / "manifests"

OUTPUT_ROOT = Path("/kaggle/working/results/experiments")
PREVIOUS_EXPERIMENT_SOURCE = None

# 'resnet18_covidqu', 'resnet18_imagenet_covidqu', 'resnet18_covidqu_syn', 'resnet18_imagenet_covidqu_syn'
EXPERIMENT_ID = "resnet18_imagenet_covidqu_syn"

PRETRAIN_EPOCHS = 70
FINETUNE_EPOCHS = None

print("LABELLED_SOURCE:", LABELLED_SOURCE, LABELLED_SOURCE.exists())
print("UNLABELLED_SOURCE:", UNLABELLED_SOURCE, UNLABELLED_SOURCE.exists())
print("SYNTHETIC_SOURCE:", SYNTHETIC_SOURCE, SYNTHETIC_SOURCE.exists())
print("MANIFEST_SOURCE:", MANIFEST_SOURCE, MANIFEST_SOURCE.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

LABELLED_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232 True
UNLABELLED_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16934 True
SYNTHETIC_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan True
MANIFEST_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests True
OUTPUT_ROOT: /kaggle/working/results/experiments
EXPERIMENT_ID: resnet18_imagenet_covidqu_syn


## 5. Link Data and Prepare Manifests

In [23]:
def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked', target, '->', source)
    else:
        print('WARNING: source missing:', source)

%cd {REPO_ROOT}
replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/unlabelled_16934', UNLABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_SOURCE)

manifest_dir = REPO_ROOT / 'data/manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
    src = MANIFEST_SOURCE / name
    dst = manifest_dir / name
    if src.exists():
        shutil.copy2(src, dst)
        print('Copied manifest:', dst)
    elif dst.exists():
        print('Using repo manifest:', dst)
    else:
        print('WARNING: missing manifest:', src)

# Normalize synthetic manifest for Kaggle. Stage 1 manifests made on Colab may contain Drive absolute paths.
src_syn_manifest = MANIFEST_SOURCE / 'synthetic_dcgan.csv'
dst_syn_manifest = manifest_dir / 'synthetic_dcgan.csv'
if src_syn_manifest.exists():
    df = pd.read_csv(src_syn_manifest)
    normalized_paths = []
    for _, row in df.iterrows():
        original = Path(str(row['image_path']))
        class_name = row['class_name']
        filename = original.name
        candidates = [
            Path('data/processed/synthetic_dcgan') / class_name / 'images' / filename,
            Path('data/processed/synthetic_dcgan') / class_name / filename,
            Path('data/processed/synthetic_dcgan') / original.name,
        ]
        selected = candidates[0]
        for candidate in candidates:
            if (REPO_ROOT / candidate).exists():
                selected = candidate
                break
        normalized_paths.append(str(selected))
    df['image_path'] = normalized_paths
    df.to_csv(dst_syn_manifest, index=False)
    print('Wrote Kaggle-normalized synthetic manifest:', dst_syn_manifest)
elif dst_syn_manifest.exists():
    print('Using repo synthetic manifest:', dst_syn_manifest)
else:
    rows = []
    class_to_label = {'COVID': 0, 'Lung_Opacity': 1, 'Viral_Pneumonia': 2, 'Normal': 3}
    image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    if SYNTHETIC_SOURCE.exists():
        for class_name, label in class_to_label.items():
            class_dir = REPO_ROOT / 'data/processed/synthetic_dcgan' / class_name
            search_root = class_dir / 'images' if (class_dir / 'images').exists() else class_dir
            for image_path in sorted(search_root.rglob('*')):
                if image_path.is_file() and image_path.suffix.lower() in image_exts:
                    rows.append({
                        'image_path': str(image_path.relative_to(REPO_ROOT)),
                        'class_name': class_name,
                        'label': label,
                        'source': 'synthetic',
                        'generator': 'dcgan',
                    })
    if rows:
        pd.DataFrame(rows).to_csv(dst_syn_manifest, index=False)
        print('Generated synthetic manifest from Kaggle synthetic folder:', dst_syn_manifest, 'rows=', len(rows))
    else:
        print('WARNING: synthetic_dcgan.csv not found and synthetic images could not be discovered. Synthetic experiments will fail until this is provided.')

!find data -maxdepth 3 -type d | sort | head -40
!ls -lh data/manifests

/kaggle/working/contrastive-synthesis-medcls_CVProject
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/labelled_4232 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/unlabelled_16934 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16934
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/train.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/val.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/test.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests

## 6. Verify Inputs and Scripts

In [24]:
SYNTHETIC_MANIFEST = REPO_ROOT / 'data/manifests/synthetic_dcgan.csv'

!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py
!python scripts/run_simclr_resnet.py --help | grep resume || true


Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

## 7. Experiment Settings: resnet18_imagenet_covidqu_syn

`PRETRAIN_EPOCHS` is the total target epoch. Increase it from 10 to 20, 30, and so on to resume in chunks.


In [25]:
EXP = 'resnet18_imagenet_covidqu_syn'
CONFIG = 'configs/experiments/resnet18/imagenet_covidqu_syn.yaml'
USES_SYNTHETIC = True
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'

pretrain_epoch_arg = f'--epochs {PRETRAIN_EPOCHS}' if PRETRAIN_EPOCHS is not None else ''
finetune_epoch_arg = f'--epochs {FINETUNE_EPOCHS}' if FINETUNE_EPOCHS is not None else ''

print('EXP:', EXP)
print('CONFIG:', CONFIG)
print('OUT:', OUT)
print('CKPT:', CKPT, 'exists=', CKPT.exists())
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())
print('USES_SYNTHETIC:', USES_SYNTHETIC)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)


EXP: resnet18_imagenet_covidqu_syn
CONFIG: configs/experiments/resnet18/imagenet_covidqu_syn.yaml
OUT: /kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn
CKPT: /kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/best_simclr_backbone.pth exists= True
RESUME_CKPT: /kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth exists= True
USES_SYNTHETIC: True
pretrain_epoch_arg: --epochs 70
finetune_epoch_arg: 


## 8. Optional: Restore Previous Kaggle Result

If you uploaded a previous output folder as a Kaggle Dataset, restore it before pretraining so SimCLR can resume.


In [ ]:
if PREVIOUS_EXPERIMENT_SOURCE is not None:
    previous = Path(PREVIOUS_EXPERIMENT_SOURCE)
    if not previous.exists():
        raise FileNotFoundError(f'PREVIOUS_EXPERIMENT_SOURCE does not exist: {previous}')
    if OUT.exists():
        shutil.rmtree(OUT)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(previous, OUT)
    print('Restored previous result folder:', previous, '->', OUT)
else:
    print('No previous result folder configured. Starting from existing /kaggle/working output if present, otherwise fresh.')

print('Resume checkpoint exists:', RESUME_CKPT.exists(), RESUME_CKPT)


## 9. Pretrain Only

Run this cell repeatedly by increasing `PRETRAIN_EPOCHS`. If it prints `Epoch 1/...` when you expected resume, stop and check `RESUME_CKPT`.


In [31]:
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())
!python scripts/run_simclr_resnet.py \
  --config "{CONFIG}" \
  --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
  --output-dir "{OUT}" \
  --resume-checkpoint "{RESUME_CKPT}" \
  {pretrain_epoch_arg}


RESUME_CKPT: /kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth exists= True
Resuming SimCLR from /kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth at epoch 35
Epoch 36/70 simclr_loss=3.1335
Epoch 37/70 simclr_loss=3.1269
Epoch 38/70 simclr_loss=3.1335
Epoch 39/70 simclr_loss=3.1290
Epoch 40/70 simclr_loss=3.1334
Epoch 41/70 simclr_loss=3.1244
Epoch 42/70 simclr_loss=3.1167
Epoch 43/70 simclr_loss=3.1192
Epoch 44/70 simclr_loss=3.1104
Epoch 45/70 simclr_loss=3.1124
Epoch 46/70 simclr_loss=3.1188
Epoch 47/70 simclr_loss=3.1156
Epoch 48/70 simclr_loss=3.1213
Epoch 49/70 simclr_loss=3.1173
Epoch 50/70 simclr_loss=3.1123
Epoch 51/70 simclr_loss=3.1087
Epoch 52/70 simclr_loss=3.1115
Epoch 53/70 simclr_loss=3.1118
Epoch 54/70 simclr_loss=3.1089
Epoch 55/70 simclr_loss=3.0973
Epoch 56/70 simclr_loss=3.1001
Epoch 57/70 simclr_loss=3.1097
Epoch 58/70 simclr_loss=3.0993


## 10. Fine-Tune Only

Run after pretraining reaches the epoch target you want to report.


In [36]:
print('CKPT:', CKPT, 'exists=', CKPT.exists())
if not CKPT.exists():
    raise FileNotFoundError(f'SimCLR checkpoint not found: {CKPT}. Finish pretraining first.')
!python scripts/run_classification_resnet.py \
  --config "{CONFIG}" \
  --manifest-dir data/manifests \
  --output-dir "{OUT}" \
  --pretrained-checkpoint "{CKPT}" \
  {finetune_epoch_arg}


CKPT: /kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/best_simclr_backbone.pth exists= True
Missing keys after SimCLR encoder load: ['fc.weight', 'fc.bias']
Epoch 1/70 train_loss=1.0566 val_loss=0.8364 val_acc=0.7150 val_f1_macro=0.5891
Epoch 2/70 train_loss=0.7323 val_loss=0.6328 val_acc=0.7957 val_f1_macro=0.7379
Epoch 3/70 train_loss=0.5837 val_loss=0.5289 val_acc=0.8361 val_f1_macro=0.8091
Epoch 4/70 train_loss=0.4906 val_loss=0.4639 val_acc=0.8432 val_f1_macro=0.8204
Epoch 5/70 train_loss=0.4323 val_loss=0.4278 val_acc=0.8527 val_f1_macro=0.8426
Epoch 6/70 train_loss=0.3869 val_loss=0.3947 val_acc=0.8622 val_f1_macro=0.8550
Epoch 7/70 train_loss=0.3548 val_loss=0.3781 val_acc=0.8694 val_f1_macro=0.8694
Epoch 8/70 train_loss=0.3344 val_loss=0.3618 val_acc=0.8694 val_f1_macro=0.8662
Epoch 9/70 train_loss=0.2950 val_loss=0.3493 val_acc=0.8741 val_f1_macro=0.8729
Epoch 10/70 train_loss=0.2815 val_loss=0.3383 val_acc=0.8812 val_f1_macro=0.8808
Epo

## 11. Display and Package Results


In [37]:
import json

metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('WARNING: metrics.json not found:', metrics_path)

!find "{OUT}" -maxdepth 4 -type f | sort
!cd /kaggle/working && zip -qr "{EXP}_results.zip" results/experiments/"{EXP}"
print('Result zip:', Path('/kaggle/working') / f'{EXP}_results.zip')


,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_imagenet_covidqu_syn,0.908665,0.919677,0.907792,0.913316,0.909857,0.908665,0.909054,47,0.91875


/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/best_checkpoint.pth
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/classification_report.csv
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/config_resolved_simclr.yaml
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/config_resolved.yaml
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/confusion_matrix.png
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/metrics.json
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/best_simclr_backbone.pth
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth
/kaggle/working/results/experiments/resnet18_imagenet_covidqu_syn/pretrain/simclr_history.json
Result zip: /kaggle/working/resnet18_imagenet_covidqu_syn_results.zip
